# 02 — Análisis exploratorio de automatización

## Proyecto: HR Process Automation Scanner

El objetivo de este notebook es analizar el ranking final de oportunidades de automatización generado en la fase anterior.

En esta fase se explorarán los patrones principales del dataset final para identificar:

- procesos HR con mayor potencial de automatización;
- sistemas HR con mayor oportunidad de optimización;
- regiones y canales con mayor fricción operativa;
- distribución de prioridades de automatización;
- tipos de soluciones recomendadas;
- oportunidades con mayor ahorro operativo estimado.

Este análisis servirá como base para las visualizaciones, el storytelling ejecutivo, el modelo predictivo y la futura aplicación en Streamlit.

In [ ]:
# Importar librerías y rutas:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"
FINAL_DATA_PATH = DATA_PATH / "final"
REPORTS_PATH = PROJECT_ROOT / "reports"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("Ruta final:", FINAL_DATA_PATH)
print("Existe data/final:", FINAL_DATA_PATH.exists())

## 2.1 Carga del dataset final

En esta sección se carga el dataset final generado en el notebook anterior.

Este dataset contiene el ranking de oportunidades de automatización por unidad operativa de HR, incluyendo:

- proceso HR;
- sistema HR;
- región;
- canal de contacto;
- volumen de casos;
- score de automatización;
- prioridad de automatización;
- solución recomendada;
- horas operativas estimadas ahorrables.

In [ ]:
# Cargar dataset final:
ranking_path = FINAL_DATA_PATH / "hr_process_automation_ranking.csv"

df = pd.read_csv(ranking_path)

df.head()

In [ ]:
# Revisar dimensiones y columnas:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.columns.tolist()

In [ ]:
# Revisar tipos de datos:
df.info()

In [ ]:
# Revisar valores nulos:
missing_summary = (
    df
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "valores_nulos"})
)

missing_summary["porcentaje_nulos"] = (
    missing_summary["valores_nulos"] / len(df) * 100
).round(2)

missing_summary.sort_values("porcentaje_nulos", ascending=False)

In [ ]:
# Resumen estadístico general:
df.describe().T

## 2.2 Ajuste de canales internos HR

El dataset original provenía de un contexto de atención al cliente. Por ese motivo, incluía canales como "Social Media", que no son adecuados para interpretar procesos internos de HR Operations.

Para adaptar el análisis al contexto de People Operations, se reinterpretarán los canales originales como canales internos de servicio al empleado.

La transformación será:

- Email → Email
- Chat → HR Chat / Virtual Assistant
- Phone → Phone
- Web Form → Employee Self-Service Portal
- Social Media → Internal Collaboration Tool

Este ajuste permite que los análisis posteriores sean coherentes con un entorno realista de HR Operations.

In [ ]:
hr_channel_mapping = {
    "Email": "Email",
    "Chat": "HR Chat / Virtual Assistant",
    "Phone": "Phone",
    "Web Form": "Employee Self-Service Portal",
    "Social Media": "Internal Collaboration Tool"
}

df["hr_contact_channel"] = df["contact_channel"].map(hr_channel_mapping)

df[["contact_channel", "hr_contact_channel"]].drop_duplicates().sort_values("contact_channel")

In [ ]:
unmapped_channels = df[df["hr_contact_channel"].isna()]["contact_channel"].unique()

print("Canales sin mapear:")
print(unmapped_channels)

### Interpretación del ajuste de canales

No se encontraron canales sin mapear, por lo que todos los valores originales fueron transformados correctamente.

A partir de este punto, el análisis utilizará `hr_contact_channel` como canal interno HR, en lugar de la columna original `contact_channel`.

Esto mejora la coherencia del proyecto y evita presentar canales propios de atención al cliente como si fueran canales reales de HR Operations.

## 2.3 Distribución de prioridad de automatización

El primer análisis consiste en revisar cuántas unidades operativas fueron clasificadas como prioridad alta, media o baja.

Esto permite entender cómo se distribuye el potencial de automatización dentro del conjunto de procesos analizados.

In [ ]:
priority_distribution = (
    df["automation_priority"]
    .value_counts()
    .reset_index()
)

priority_distribution.columns = ["automation_priority", "total_units"]

priority_distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    priority_distribution["automation_priority"],
    priority_distribution["total_units"]
)

plt.title("Distribución de prioridad de automatización")
plt.xlabel("Prioridad de automatización")
plt.ylabel("Número de unidades operativas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Interpretación

La distribución de prioridad muestra una segmentación equilibrada de las unidades operativas analizadas.

La clasificación permite separar las oportunidades en tres grupos:

- prioridad alta: mejores candidatos relativos para automatización;
- prioridad media: oportunidades con potencial, pero no necesariamente inmediatas;
- prioridad baja: unidades con menor prioridad de intervención.

Esta segmentación es útil para priorizar iniciativas de automatización de forma más ordenada.

## 2.4 Top 10 oportunidades de automatización

Este análisis muestra las unidades operativas con mayor Automation Opportunity Score.

Estas unidades representan los mejores candidatos relativos para iniciativas de automatización u optimización con IA.

In [ ]:
top_10_opportunities = df[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "hr_contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].head(10)

top_10_opportunities

In [ ]:
top_10_plot = top_10_opportunities.sort_values("automation_score", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    top_10_plot["hr_process_name"] + " | " + top_10_plot["region"],
    top_10_plot["automation_score"]
)

plt.title("Top 10 oportunidades por Automation Opportunity Score")
plt.xlabel("Automation Opportunity Score")
plt.ylabel("Proceso HR y región")
plt.tight_layout()
plt.show()

### Interpretación

El ranking muestra que las mejores oportunidades no dependen únicamente del proceso HR general.

Las unidades mejor posicionadas combinan proceso, sistema, región y canal interno. Esto confirma que la automatización debe analizarse a nivel operativo granular, no solo a nivel de proceso amplio.

El recomendador permite identificar oportunidades concretas donde una intervención tecnológica podría tener mayor impacto.

## 2.5 Procesos HR con mayor score promedio

Este análisis permite identificar qué procesos de HR Operations presentan mayor potencial promedio de automatización.

In [ ]:
process_score_summary = (
    df
    .groupby("hr_process_name")
    .agg(
        total_units=("process_unit_id_v2", "count"),
        total_cases=("total_cases", "sum"),
        avg_automation_score=("automation_score", "mean"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum")
    )
    .reset_index()
    .sort_values("avg_automation_score", ascending=False)
)

process_score_summary

In [ ]:
process_score_plot = process_score_summary.sort_values("avg_automation_score", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    process_score_plot["hr_process_name"],
    process_score_plot["avg_automation_score"]
)

plt.title("Score promedio de automatización por proceso HR")
plt.xlabel("Automation Opportunity Score promedio")
plt.ylabel("Proceso HR")
plt.tight_layout()
plt.show()

### Interpretación

Los scores promedio por proceso son relativamente similares. Esto indica que el potencial de automatización no depende únicamente del proceso HR general.

La diferencia real aparece al analizar combinaciones específicas de:

- proceso HR;
- sistema HR;
- región;
- canal interno.

Por este motivo, el valor principal del recomendador está en identificar unidades operativas concretas, no solo procesos generales.

## 2.6 Procesos con mayor ahorro operativo estimado

Este análisis identifica los procesos donde la automatización podría liberar más horas operativas estimadas.

El objetivo es priorizar procesos no solo por score, sino también por impacto operativo potencial.

In [ ]:
process_savings_summary = (
    df
    .groupby("hr_process_name")
    .agg(
        total_cases=("total_cases", "sum"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum"),
        avg_automation_score=("automation_score", "mean")
    )
    .reset_index()
    .sort_values("total_estimated_hours_saved", ascending=False)
)

process_savings_summary

In [ ]:
process_savings_plot = process_savings_summary.sort_values("total_estimated_hours_saved", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    process_savings_plot["hr_process_name"],
    process_savings_plot["total_estimated_hours_saved"]
)

plt.title("Horas operativas estimadas ahorrables por proceso HR")
plt.xlabel("Horas operativas estimadas ahorrables")
plt.ylabel("Proceso HR")
plt.tight_layout()
plt.show()

### Interpretación

El análisis de ahorro operativo estimado permite identificar dónde una iniciativa de automatización podría liberar mayor capacidad operativa.

Este enfoque complementa el Automation Opportunity Score, ya que una unidad puede tener buen score, pero menor impacto absoluto si su volumen o tiempo operativo estimado es bajo.

Por tanto, la priorización final debe considerar tanto el score como el ahorro operativo estimado.

## 2.7 Sistemas HR con mayor potencial de automatización

Este análisis permite identificar qué plataformas o sistemas HR concentran más oportunidades de automatización.

Esto es útil para priorizar iniciativas tecnológicas, integraciones, mejoras de workflow o automatizaciones dentro del ecosistema HRIS.

In [ ]:
system_score_summary = (
    df
    .groupby("hr_system_name")
    .agg(
        total_units=("process_unit_id_v2", "count"),
        total_cases=("total_cases", "sum"),
        avg_automation_score=("automation_score", "mean"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum")
    )
    .reset_index()
    .sort_values("total_estimated_hours_saved", ascending=False)
)

system_score_summary

In [ ]:
system_plot = system_score_summary.sort_values("total_estimated_hours_saved", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    system_plot["hr_system_name"],
    system_plot["total_estimated_hours_saved"]
)

plt.title("Horas operativas estimadas ahorrables por sistema HR")
plt.xlabel("Horas operativas estimadas ahorrables")
plt.ylabel("Sistema HR")
plt.tight_layout()
plt.show()

### Interpretación

El análisis por sistema permite identificar qué plataformas concentran mayor potencial de ahorro operativo.

Esta lectura es útil para priorizar mejoras tecnológicas, automatización de workflows, integración de sistemas o rediseño de procesos dentro del ecosistema HR.

Desde una perspectiva de transformación digital, este análisis ayuda a decidir dónde intervenir primero dentro del stack tecnológico de HR.

## 2.8 Regiones con mayor oportunidad de automatización

Este análisis permite identificar en qué regiones se concentra mayor potencial de ahorro operativo.

La lectura regional ayuda a priorizar iniciativas de automatización por mercado, geografía o centro de servicios.

In [ ]:
region_summary = (
    df
    .groupby("region")
    .agg(
        total_units=("process_unit_id_v2", "count"),
        total_cases=("total_cases", "sum"),
        avg_automation_score=("automation_score", "mean"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum")
    )
    .reset_index()
    .sort_values("total_estimated_hours_saved", ascending=False)
)

region_summary

In [ ]:
region_plot = region_summary.sort_values("total_estimated_hours_saved", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(
    region_plot["region"],
    region_plot["total_estimated_hours_saved"]
)

plt.title("Horas operativas estimadas ahorrables por región")
plt.xlabel("Horas operativas estimadas ahorrables")
plt.ylabel("Región")
plt.tight_layout()
plt.show()

### Interpretación

El análisis regional permite observar dónde se concentra el mayor potencial de optimización operativa.

Esta lectura es útil para diseñar una estrategia de despliegue gradual, empezando por las regiones con mayor volumen, mayor fricción o mayor ahorro operativo estimado.

En un contexto real de HR Operations, esta información podría apoyar decisiones sobre priorización regional, asignación de recursos o implementación progresiva de automatizaciones.

## 2.9 Canales internos con mayor fricción operativa

Este análisis evalúa qué canales internos concentran mayor oportunidad de automatización.

Los canales como email, chat interno, portal de autoservicio o herramientas colaborativas pueden requerir soluciones distintas:

- workflows;
- chatbots;
- asistentes virtuales;
- reglas de enrutamiento;
- automatización parcial;
- alertas SLA.

In [ ]:
channel_summary = (
    df
    .groupby("hr_contact_channel")
    .agg(
        total_units=("process_unit_id_v2", "count"),
        total_cases=("total_cases", "sum"),
        avg_automation_score=("automation_score", "mean"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum")
    )
    .reset_index()
    .sort_values("total_estimated_hours_saved", ascending=False)
)

channel_summary

In [ ]:
channel_plot = channel_summary.sort_values("total_estimated_hours_saved", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(
    channel_plot["hr_contact_channel"],
    channel_plot["total_estimated_hours_saved"]
)

plt.title("Horas operativas estimadas ahorrables por canal interno HR")
plt.xlabel("Horas operativas estimadas ahorrables")
plt.ylabel("Canal interno HR")
plt.tight_layout()
plt.show()

### Interpretación

El análisis por canal interno permite identificar dónde se concentra mayor fricción operativa en la entrada y gestión de solicitudes HR.

Este análisis es especialmente útil porque diferentes canales requieren diferentes tipos de intervención:

- los portales de autoservicio pueden mejorarse con workflows y formularios inteligentes;
- los chats internos pueden apoyarse con asistentes virtuales;
- el email puede requerir clasificación automática, reglas de enrutamiento o extracción de información;
- las herramientas colaborativas internas pueden requerir integración con sistemas HRIS o automatización de seguimiento.

Por tanto, el canal no solo describe cómo entra el caso, sino también qué tipo de solución tecnológica puede ser más adecuada.

## 2.10 Distribución de soluciones recomendadas

Este análisis muestra qué tipos de soluciones de automatización aparecen con mayor frecuencia.

El objetivo es entender si las oportunidades detectadas se orientan más hacia workflows, chatbots, alertas SLA, rediseño de procesos o revisión humana asistida.

In [ ]:
solution_summary = (
    df
    .groupby("recommended_solution")
    .agg(
        total_units=("process_unit_id_v2", "count"),
        total_cases=("total_cases", "sum"),
        avg_automation_score=("automation_score", "mean"),
        total_estimated_hours_saved=("estimated_operational_hours_saved", "sum")
    )
    .reset_index()
    .sort_values("total_estimated_hours_saved", ascending=False)
)

solution_summary

In [ ]:
solution_plot = solution_summary.sort_values("total_units", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    solution_plot["recommended_solution"],
    solution_plot["total_units"]
)

plt.title("Distribución de soluciones recomendadas")
plt.xlabel("Número de unidades operativas")
plt.ylabel("Solución recomendada")
plt.tight_layout()
plt.show()

### Interpretación

La distribución de soluciones recomendadas traduce el análisis en acciones concretas.

En lugar de limitarse a identificar procesos problemáticos, el recomendador propone posibles líneas de intervención, como:

- workflow automation;
- chatbot o asistente virtual;
- automatización parcial con alertas SLA;
- rediseño del proceso antes de automatizar;
- revisión humana asistida.

Esto refuerza el valor del proyecto porque conecta análisis de datos con decisiones prácticas de transformación digital.

## 2.11 Matriz de priorización final

La matriz de priorización combina score de automatización, ahorro operativo estimado, prioridad y tipo de solución recomendada.

Esta tabla resume las mejores oportunidades detectadas y sirve como base para la toma de decisiones.

In [ ]:
final_priority_matrix = df[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "hr_contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].copy()

final_priority_matrix.head(20)

### Interpretación

La matriz de priorización final resume las oportunidades más relevantes para una estrategia de automatización en HR Operations.

Esta tabla permite responder preguntas clave de negocio:

- qué unidad operativa debería priorizarse;
- qué proceso HR está involucrado;
- qué sistema debería intervenirse;
- en qué región aparece la oportunidad;
- por qué canal interno entra la solicitud;
- cuál es el score de automatización;
- qué solución se recomienda;
- cuánto ahorro operativo estimado podría generarse.

Esta matriz será uno de los principales activos del proyecto, ya que puede alimentar tanto una presentación ejecutiva como una aplicación interactiva en Streamlit.

## 2.12 Exportación de resultados finales del EDA

En esta sección se guardan los resultados principales del análisis exploratorio.

Estos archivos serán utilizados en fases posteriores del proyecto, incluyendo:

- visualizaciones ejecutivas;
- modelo de Machine Learning;
- aplicación en Streamlit;
- presentación final;
- documentación del repositorio.

In [ ]:
system_score_summary.to_csv(REPORTS_PATH / "system_score_summary.csv", index=False)
region_summary.to_csv(REPORTS_PATH / "region_summary.csv", index=False)
channel_summary.to_csv(REPORTS_PATH / "channel_summary.csv", index=False)
solution_summary.to_csv(REPORTS_PATH / "solution_summary.csv", index=False)
final_priority_matrix.to_csv(REPORTS_PATH / "final_priority_matrix.csv", index=False)

print("Archivos finales del EDA guardados correctamente en la carpeta reports.")

In [ ]:
df.to_csv(
    FINAL_DATA_PATH / "hr_process_automation_ranking_clean.csv",
    index=False
)

print("Dataset final corregido guardado correctamente en data/final.")

In [ ]:
print("Archivos guardados en reports:")

for file_path in REPORTS_PATH.glob("*.csv"):
    print(file_path.name)

print("\nArchivos guardados en data/final:")

for file_path in FINAL_DATA_PATH.glob("*.csv"):
    print(file_path.name)

## 2.13 Conclusiones del análisis exploratorio

En este notebook se analizó el ranking final de oportunidades de automatización generado en la fase anterior.

El análisis permitió identificar patrones relevantes para la toma de decisiones:

- la clasificación de prioridad de automatización quedó distribuida en tres grupos: alta, media y baja;
- las oportunidades más relevantes no aparecen únicamente a nivel de proceso general, sino en unidades operativas específicas;
- el potencial de automatización depende de la combinación entre proceso HR, sistema, región y canal interno de contacto;
- los procesos con mayor ahorro operativo estimado permiten priorizar iniciativas con mayor impacto potencial;
- las soluciones recomendadas permiten traducir el análisis en acciones concretas, como workflow automation, chatbot, automatización parcial con alertas SLA o rediseño del proceso.

Durante el análisis también se realizó un ajuste de coherencia de negocio sobre los canales de contacto. El dataset original provenía de un contexto de atención al cliente e incluía canales como "Social Media". Para adaptarlo al contexto de HR Operations, este canal fue reinterpretado como "Internal Collaboration Tool", representando herramientas internas como Slack, Microsoft Teams o canales corporativos de colaboración.

Este ajuste permite que el análisis sea más realista desde una perspectiva de People Operations, ya que los procesos internos de HR normalmente se gestionan mediante email, portales de autoservicio, chat interno, teléfono o herramientas colaborativas internas.

Una conclusión importante es que el valor del recomendador no está en decir simplemente qué proceso automatizar, sino en identificar dónde, bajo qué condiciones y con qué tipo de solución conviene intervenir.

Los archivos generados en este notebook serán utilizados como base para la siguiente fase del proyecto: construcción de un modelo de Machine Learning para predecir la prioridad de automatización.

## Exportación específica para Tableau

En esta sección se genera un archivo CSV simplificado y limpio para construir el dashboard ejecutivo en Tableau.

In [ ]:
# Verificar dataframes disponibles para exportar
print("Columnas de df:")
print(df.columns.tolist())

In [ ]:
# Crear archivo específico para Tableau
tableau_export = df[
    [
        "ranking",
        "hr_process_name",
        "hr_system_name",
        "region",
        "hr_contact_channel",
        "total_cases",
        "automation_score",
        "automation_priority",
        "recommended_solution",
        "estimated_operational_hours_saved"
    ]
].copy()

tableau_export.to_csv(
    FINAL_DATA_PATH / "tableau_hr_automation_dashboard.csv",
    index=False,
    encoding="utf-8-sig",
    sep=",",
    decimal=".",
)

print("Archivo exportado correctamente para Tableau.")
print(FINAL_DATA_PATH / "tableau_hr_automation_dashboard.csv")

In [ ]:
print(df[['automation_score']].head(10))
print(df['automation_score'].isna().sum())